In [5]:
import json

file_path = 'TRECIS-CTIT-H-extracted-merged.json'

# 使用列表推导式直接生成列表
with open(file_path, 'r', encoding='utf-8') as f:
    # strip() 移除行尾换行符，if line 判断避免空行报错
    data_list = [json.loads(line) for line in f if line.strip()]

# 验证结果
print(f"转换完成！数据类型: {type(data_list)}")
print(f"总共有 {len(data_list)} 个元素")

转换完成！数据类型: <class 'list'>
总共有 2458 个元素


In [6]:
first_row = data_list[0]  # 获取第一行数据（字典）
print(first_row.keys())   # 查看这一行有哪些字段

dict_keys(['id', 'created_at', 'place', 'coordinates', 'text', 'geo', 'user'])


In [7]:
import pandas as pd
import re
import json

# 1. 加载数据
file_path = 'TRECIS-CTIT-H-extracted-merged.json'
data = []
with open(file_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            data.append(json.loads(line))

df = pd.DataFrame(data)

# 2. 定义清洗函数
def clean_and_filter(text):
    if not isinstance(text, str):
        return None
    
    # 删除 URL (匹配 http, https, www 开始的字符)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    
    # 移除多余空格 (将多个空格/换行符替换为单个空格，并去除首尾空格)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 过滤掉单词量少于 3 的句子
    # 注意：这里以空格拆分单词，如果是中文可能需要另外的切词逻辑
    if len(text.split()) < 3:
        return None
        
    return text

# 3. 执行清洗
# 我们创建一个新列 'cleaned_text'，然后删掉清洗后变为 None 的行
df['cleaned_text'] = df['text'].apply(clean_and_filter)
df_result = df.dropna(subset=['cleaned_text']).copy()

# 查看结果
print(f"原始数据量: {len(df)}")
print(f"清洗后数据量: {len(df_result)}")
print("\n--- 清洗效果预览 ---")
print(df_result[['text', 'cleaned_text']].head())

# 如果需要保存结果
# df_result.to_json('cleaned_data.json', orient='records', force_ascii=False, lines=True)

原始数据量: 2458
清洗后数据量: 2453

--- 清洗效果预览 ---
                                                text  \
0  #colorado. Told you its #amazing http://t.co/6...   
1  RT @northfortynews: Tanker helicopter heads up...   
2  #Evacuation center Cache La Poudre Middle Scho...   
3  20F degrees cooler tomorrow in North Central &...   
4  FEMA has authorized the use of federal funds t...   

                                        cleaned_text  
0                   #colorado. Told you its #amazing  
1  RT @northfortynews: Tanker helicopter heads up...  
2  #Evacuation center Cache La Poudre Middle Scho...  
3  20F degrees cooler tomorrow in North Central &...  
4  FEMA has authorized the use of federal funds t...  


In [8]:
import pandas as pd
import json
import os
from openai import OpenAI
import time

# 1. 初始化客户端 (确保你已设置环境变量或直接填写 API Key)
client = OpenAI(api_key="your LLM API")

def analyze_tweet_geospatial(text):
    """调用 GPT-4o-mini 分析单条推文的空间信息"""
    system_prompt = """
    ### Role: Strict Geospatial Entity Extractor
    
    ### Task: 
    Extract ONLY high-value, explicit geographical entities from the provided text.
    
    ### Rules:
    1. VALUABLE ENTITIES TO EXTRACT:
       - Extract SPECIFIC, NAMED locations (e.g., cities like "Colorado", parks like "High Park", streets, landmarks).
       - COMPOSITE NAMES ARE REQUIRED: If generic words have a specific name attached, extract the WHOLE named entity (e.g., "Rocky Mountain", "Mississippi River", "Hilton Hotel", "Main Street").
       - Split hashtags like #HighParkFire to extract only its specific location component ("High Park").
       - Treat specific intersections (e.g., "Mulberry & Lemay") as a SINGLE entity.
    
    2. STRICT FORBIDDEN (FILTER OUT GENERIC NOUNS):
       - DO NOT extract STANDALONE, UNNAMED generic nouns that lack any specific names or identifiers (e.g., do NOT extract words like "a mountain", "the hotel", "a river", "the street", "the city", "a hospital" when they are used globally without a name).
       - DO NOT infer or guess any locations. Only extract what is literally written.
       - DO NOT extract non-geographical entities like people, weather, or time.
    
    ### Output Format (JSON):
    - If valuable geographical entities are found: 
      {"entities": ["Location A", "Location B"]}
    - If NO valuable geographical info is found: 
      {"result": "unknown"}
    """
    
    try:
        response = client.chat.completions.create(
            model="gpt-5.4-mini",
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Text: {text}"}
            ],
            response_format={ "type": "json_object" }, # 强制 JSON
            temperature=0 # 降低随机性，保证推理的一致性
        )
        return json.loads(response.choices[0].message.content)
    except Exception as e:
        print(f"Error processing: {e}")
        return {"entities": [], "spatial_relations": [], "error": str(e)}

# 2. 读取之前清洗好的数据
# 假设你的变量名是 df_cleaned
# 如果数据量大，建议先取前 10 条测试：test_df = df_cleaned.head(10).copy()
df_to_process = df_result.copy()

results = []
print(f"Starting API calls for {len(df_to_process)} rows...")

# 3. 循环处理并保存（增加简单的批次保存逻辑）
for index, row in df_to_process.iterrows():
    res = analyze_tweet_geospatial(row['cleaned_text'])
    results.append(res)
    
    # 每处理 50 条打印一次进度
    if len(results) % 50 == 0:
        print(f"Processed {len(results)}/{len(df_to_process)} rows...")

# 4. 将结果合并回原 DataFrame
df_to_process['llm_analysis'] = results

# 5. 展开 JSON 结果（可选：方便论文后续分析）
# 将 entities 和 relations 提取为单独的列，或者直接保存为大 CSV/Excel
df_to_process.to_csv('experiment_onlyentityresults_mini2.csv', index=False, encoding='utf-8-sig')

print("Experiment data saved to 'experiment_onlyentityresults_mini2.csv'")

Starting API calls for 2453 rows...
Processed 50/2453 rows...
Processed 100/2453 rows...
Processed 150/2453 rows...
Processed 200/2453 rows...
Processed 250/2453 rows...
Processed 300/2453 rows...
Processed 350/2453 rows...
Processed 400/2453 rows...
Processed 450/2453 rows...
Processed 500/2453 rows...
Processed 550/2453 rows...
Processed 600/2453 rows...
Processed 650/2453 rows...
Processed 700/2453 rows...
Processed 750/2453 rows...
Processed 800/2453 rows...
Processed 850/2453 rows...
Processed 900/2453 rows...
Processed 950/2453 rows...
Processed 1000/2453 rows...
Processed 1050/2453 rows...
Processed 1100/2453 rows...
Processed 1150/2453 rows...
Processed 1200/2453 rows...
Processed 1250/2453 rows...
Processed 1300/2453 rows...
Processed 1350/2453 rows...
Processed 1400/2453 rows...
Processed 1450/2453 rows...
Processed 1500/2453 rows...
Processed 1550/2453 rows...
Processed 1600/2453 rows...
Processed 1650/2453 rows...
Processed 1700/2453 rows...
Processed 1750/2453 rows...
Proc

In [9]:
import pandas as pd
import ast

INPUT_CSV = "experiment_onlyentityresults_mini2.csv"

OUTPUT_CSV = "experiment_onlyentityresults_mini3.csv"

# =====================================================
# LOAD CSV
# =====================================================

df = pd.read_csv(INPUT_CSV)

print("Original Rows:", len(df))

# =====================================================
# FILTER FUNCTION
# =====================================================

def is_valid_llm_analysis(x):

    # ---------------------------------------------
    # empty
    # ---------------------------------------------

    if pd.isna(x):
        return False

    # ---------------------------------------------
    # parse llm_analysis
    # ---------------------------------------------

    try:

        data = ast.literal_eval(str(x))

        # -----------------------------------------
        # result == unknown
        # -----------------------------------------

        result = data.get("result")

        if result == "unknown":
            return False

        # -----------------------------------------
        # entities empty
        # -----------------------------------------

        entities = data.get("entities", [])

        if len(entities) == 0:
            return False

        return True

    except:

        return False

# =====================================================
# REMOVE INVALID ROWS
# =====================================================

df_clean = df[

    df["llm_analysis"].apply(is_valid_llm_analysis)

]

print("Clean Rows:", len(df_clean))

print("Removed Rows:", len(df) - len(df_clean))

# =====================================================
# SAVE
# =====================================================

df_clean.to_csv(

    OUTPUT_CSV,

    index=False
)

print("Saved:", OUTPUT_CSV)

Original Rows: 2453
Clean Rows: 2047
Removed Rows: 406
Saved: experiment_onlyentityresults_mini3.csv
